# ReceiptGuard-ML Training on Kaggle

This notebook trains the LayoutLM-based receipt field extraction model on Kaggle GPU.

## Setup

1. **Data**: Uses SROIE2019 dataset from Kaggle
2. **Model**: LayoutLM-base-uncased fine-tuned for NER
3. **Hardware**: Kaggle GPU (T4/P100)

## Fixed Issues

- ✅ Fixed logits variable error in training loop
- ✅ Fixed config path references (training.seed, training.output_dir, etc.)
- ✅ Fixed data parsing indentation bug
- ✅ Fixed model config creation
- ✅ Added missing dependencies (sentencepiece, tiktoken)

In [ ]:
# Install dependencies
!pip install transformers torch torchvision pillow sentencepiece tiktoken
!pip install huggingface_hub accelerate

In [ ]:
# Clone the repository with all fixes
!git clone https://github.com/MoeenUddin01/Receipt_Guard.git
%cd Receipt_Guard

In [ ]:
# Setup paths for Kaggle environment
import sys
import os
from pathlib import Path

# Add project to Python path
sys.path.append('/kaggle/working/Receipt_Guard')

# Create Kaggle-specific config overrides
kaggle_overrides = {
    # Data paths for Kaggle
    'paths.raw_data_dir': '/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019',
    'paths.processed_data_dir': '/kaggle/working/processed',
    'paths.artifacts_dir': '/kaggle/working/artifacts',
    'paths.checkpoints_dir': '/kaggle/working/artifacts/checkpoints',
    'paths.evaluation_dir': '/kaggle/working/artifacts/evaluation',
    'paths.logs_dir': '/kaggle/working/artifacts/logs',
    
    # Model path - use HuggingFace model for Kaggle
    'model.model_path': 'microsoft/layoutlm-base-uncased',
    
    # Training parameters optimized for Kaggle GPU
    'training.num_epochs': 15,
    'training.batch_size': 16,  # Optimized for Kaggle GPU
    'training.learning_rate': 5e-5,
    'training.weight_decay': 0.01,
    'training.warmup_ratio': 0.1,
    'training.max_grad_norm': 1.0,
    'training.seed': 42,
    
    # Data parameters
    'data.max_length': 512,
    
    # Model parameters
    'model.num_labels': 9,  # O, B-COMPANY, I-COMPANY, B-DATE, I-DATE, B-ADDRESS, I-ADDRESS, B-TOTAL, I-TOTAL
    'model.dropout': 0.1
}

print("Kaggle configuration prepared:")
for key, value in kaggle_overrides.items():
    print(f"  {key}: {value}")

In [ ]:
# Import and setup configuration
from src.config import override_config, CFG
from src.pipelines.model_training_pipeline import run_training_pipeline

# Apply Kaggle overrides
override_config(kaggle_overrides)

print("Configuration loaded and overridden for Kaggle environment")
print(f"Model path: {CFG.model.model_path}")
print(f"Data path: {CFG.paths.raw_data_dir}")
print(f"Training epochs: {CFG.training.num_epochs}")
print(f"Batch size: {CFG.training.batch_size}")

In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available - using CPU")

In [ ]:
# Check data availability
data_path = Path('/kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019')
if data_path.exists():
    print(f"Data directory found: {data_path}")
    print("Contents:")
    for item in data_path.iterdir():
        if item.is_dir():
            file_count = len(list(item.iterdir()))
            print(f"  📁 {item.name}/ ({file_count} files)")
        else:
            print(f"  📄 {item.name}")
else:
    print(f"Data directory NOT found: {data_path}")
    print("Please add SROIE2019 dataset to this notebook")

## Start Training

The training pipeline will:
1. **Setup**: Create directories, set random seed, detect device
2. **Data**: Load and tokenize SROIE2019 dataset
3. **Model**: Build LayoutLM model with classification head
4. **Train**: Fine-tune the model with mixed precision
5. **Evaluate**: Calculate validation metrics
6. **Save**: Store best model checkpoint

In [ ]:
# Run the complete training pipeline
print("=" * 80)
print("STARTING RECEIPTGUARD-ML TRAINING ON KAGGLE")
print("=" * 80)

try:
    # Run training with all the fixes applied
    training_summary = run_training_pipeline(CFG)
    
    print("\n" + "=" * 80)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    
    # Display results
    print(f"\n📊 Training Summary:")
    print(f"  Final training loss: {training_summary.get('final_train_loss', 'N/A')}")
    print(f"  Final validation loss: {training_summary.get('final_val_loss', 'N/A')}")
    print(f"  Best validation loss: {training_summary.get('best_val_loss', 'N/A')}")
    print(f"  Training time: {training_summary.get('training_time', 'N/A')}")
    
    # Check if model was saved
    checkpoint_path = Path('/kaggle/working/artifacts/checkpoints/best_model.pt')
    if checkpoint_path.exists():
        print(f"\n✅ Model saved to: {checkpoint_path}")
        print(f"   File size: {checkpoint_path.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"\n❌ Model checkpoint not found at expected location")
        
except Exception as e:
    print("\n" + "=" * 80)
    print("TRAINING FAILED!")
    print("=" * 80)
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    print("\nFull traceback:")
    import traceback
    traceback.print_exc()

## Training Results

If training completed successfully, you can:
1. **Download** the model checkpoint from `/kaggle/working/artifacts/checkpoints/best_model.pt`
2. **View** training logs in `/kaggle/working/artifacts/logs/`
3. **Analyze** evaluation metrics in `/kaggle/working/artifacts/evaluation/`

## Model Information

- **Architecture**: LayoutLM-base-uncased + Linear classification head
- **Task**: Named Entity Recognition for receipt fields
- **Labels**: O, B-COMPANY, I-COMPANY, B-DATE, I-DATE, B-ADDRESS, I-ADDRESS, B-TOTAL, I-TOTAL
- **Training**: Fine-tuned on SROIE2019 dataset with mixed precision